In [8]:
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
import google.generativeai as genai

C:\Users\João de Paula\AppData\Local\Temp\ipykernel_20032\4031590512.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [41]:
genai.configure(api_key="key")
configuracao = genai.GenerationConfig(
    temperature=0,       # 0.0 é mais factual/rígido, 1.0 é mais criativo
    top_p=0.9,
    max_output_tokens=2000  # Limita o tamanho da resposta
)

In [14]:
source = "data/sicredi_1779225721762.pdf"
converter = DocumentConverter()
doc = converter.convert(source).document
print(doc.export_to_markdown())

[INFO] 2026-06-01 09:04:41,916 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-01 09:04:41,918 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-01 09:04:41,934 [RapidOCR] download_file.py:60: File exists and is valid: E:\02 - repos\Gestor-money\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-01 09:04:41,935 [RapidOCR] main.py:50: Using E:\02 - repos\Gestor-money\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-01 09:04:42,295 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-01 09:04:42,297 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-01 09:04:42,300 [RapidOCR] download_file.py:60: File exists and is valid: E:\02 - repos\Gestor-money\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-01 09:04:42,302 [RapidOCR] main.py:50: Using E:\02 - repos\Gestor-money\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_m

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

<!-- image -->

Associado:

JOAO PEDRO THOMAZ BASILIO DE PAULA

Cooperativa:

0710

Conta:

10071258-0

## Extrato (Período de 04/05/2026 a 19/05/2026)

| Data       | Descrição                                        | Documento   | Valor (R$)   | Saldo (R$)   |
|------------|--------------------------------------------------|-------------|--------------|--------------|
|            | SALDO ANTERIOR                                   |             |              | 4.351,68     |
| 04/05/2026 | PAGAMENTO PIX 56833237700 JAIR DA SILVA COELHO   | PIX_DEB     | -5,00        | 4.346,68     |
| 04/05/2026 | PAGAMENTO PIX 06012414000117 OLA FLORIANO        | PIX_DEB     | -186,71      | 4.159,97     |
| 04/05/2026 | LIQUIDACAO BOLETO SICREDI 62343339000148 CENTRO  | 261003371   | -811,20      | 3.348,77     |
| 04/05/2026 | PAGTO FATURA MASTER                              | 019082035   | -887,46      | 2.461,31     |
| 04/05/2026 | PAGAMENTO PIX 60444437000146 LIGHT SERVICOS DE E | PIX_DEB    

In [48]:
system_prompt = """
Você é um microsserviço de backend especializado em parsing e estruturação de dados financeiros brutos. Sua única tarefa é analisar textos em Markdown (provenientes de conversões de PDFs bancários) e transformá-los em um JSON padronizado.

### DIRETRIZES DE VALIDAÇÃO CRÍTICA
1. Avalie se o texto fornecido é REALMENTE uma fatura de cartão de crédito ou um extrato bancário.
2. Se o texto for um documento genérico, currículo, relatório corporativo, receita médica ou qualquer coisa que NÃO seja um extrato ou fatura financeira, você deve interromper a análise imediatamente e retornar o JSON de erro configurando "documento_valido": false e "codigo_erro": "INVALID_DOCUMENT_TYPE". Todos os outros campos devem ser null ou arrays vazios.

### REGRAS DE PADRONIZAÇÃO DE DADOS
- Tipo de Documento: Deve ser estritamente "FATURA" ou "EXTRATO".
- Banco: Identifique o nome do banco emissor (ex: "Nubank", "Itaú", "Inter"). Se não identificar, use "DESCONHECIDO".
- Datas: Converta TODAS as datas para o formato ISO "YYYY-MM-DD". Se o documento não informar o ano, assuma o ano corrente (2026).
- Horas: Se houver registro de hora da transação, formate como "HH:MM:SS". Caso contrário, retorne null.
- Valores: Devem ser do tipo FLOAT puro (sem "R$", sem pontos como separador de milhar, e use ponto para decimais). 
  - Saídas/Gastos/Débitos: Devem ser OBRIGATORIAMENTE NÚMEROS NEGATIVOS (ex: -45.90).
  - Entradas/Rendimentos/Créditos/Pagamentos de Fatura: Devem ser OBRIGATORIAMENTE NÚMEROS POSITIVOS (ex: 1500.00).
- Descrição: Limpe o texto da transação eliminando caracteres especiais desnecessários ou quebras de linha estranhas vindas do PDF.
- Categoria: Se o documento fornecer uma categoria para a transação (ex: "Alimentação", "Transporte"), inclua-a. Caso contrário, retorne null, caso for obvio (mercado, posto...) infira.
- Saldo Parcial: Se o documento for um Extrato e mostrar o saldo da conta após aquela transação específica, extraia o valor. Caso contrário, retorne null.

Sua resposta deve conter APENAS o objeto JSON, sem saudações, sem explicações e sem blocos de código Markdown (não use ```json).
 """
model = genai.GenerativeModel('gemini-3.1-flash-lite',
                              system_instruction= system_prompt,)


In [49]:

contagem = model.count_tokens(doc.export_to_markdown())
print(f"O seu prompt consumirá: {contagem.total_tokens} tokens.")
response = model.generate_content(
    doc.export_to_markdown(),
    generation_config=configuracao
)
print(response.text)


O seu prompt consumirá: 1749 tokens.
{
  "documento_valido": true,
  "codigo_erro": null,
  "tipo_documento": "EXTRATO",
  "banco": "Sicredi",
  "titular": "JOAO PEDRO THOMAZ BASILIO DE PAULA",
  "periodo": {
    "data_inicio": "2026-05-04",
    "data_fim": "2026-05-19"
  },
  "transacoes": [
    {
      "data": "2026-05-04",
      "hora": null,
      "descricao": "PAGAMENTO PIX 56833237700 JAIR DA SILVA COELHO",
      "valor": -5.0,
      "categoria": "Transferência",
      "saldo_parcial": 4346.68
    },
    {
      "data": "2026-05-04",
      "hora": null,
      "descricao": "PAGAMENTO PIX 06012414000117 OLA FLORIANO",
      "valor": -186.71,
      "categoria": "Transferência",
      "saldo_parcial": 4159.97
    },
    {
      "data": "2026-05-04",
      "hora": null,
      "descricao": "LIQUIDACAO BOLETO SICREDI 62343339000148 CENTRO",
      "valor": -811.2,
      "categoria": "Pagamento de Boleto",
      "saldo_parcial": 3348.77
    },
    {
      "data": "2026-05-04",
      "hora

In [50]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True


doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

In [5]:
source = "data/sicredi_1779225777933.pdf"

doc = doc_converter.convert(source).document
fatura = doc.export_to_markdown()
fatura

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

'<!-- image -->\n\nPagamento total (R$)\n\n## R$ 887,46\n\nÉ sempre a melhor opção\n\nVencimento 10/mai\n\nNão existem débitos pendentes de sua responsabilidade pelo uso desse cartão no período de jan a dez de 2025 (lei 12.007/09). Esta declaração substitui os comprovantes mensais de 2025 e dos anos anteriores.\n\n## Parcelamento (R$)\n\nVocê não possui oferta de parcelamento. Verifique a melhor opção junto a sua cooperativa.\n\n| Resumo da fatura                           | Valor (R$)   | Valor (US$)   |\n|--------------------------------------------|--------------|---------------|\n| Total da fatura anterior                   | 0,00         |               |\n| Pagamentos &#124; Créditos                 | 0,00         | 0,00          |\n| Despesas atuais &#124; Débitos no Brasil   | 887,46       |               |\n| Despesas atuais &#124; Débitos no Exterior | 0,00         | 0,00          |\n| Subtotal Despesas e Pagamentos             | 887,46       |               |\n| Encargos rot

In [51]:
contagem = model.count_tokens(fatura)

# 5. Exiba o resultado
print(f"O seu prompt consumirá: {contagem.total_tokens} tokens.")
response = model.generate_content(
    fatura,
    generation_config=configuracao
)
print(response.text)


O seu prompt consumirá: 3929 tokens.
{
  "documento_valido": true,
  "codigo_erro": null,
  "tipo_documento": "FATURA",
  "banco": "Sicredi",
  "data_vencimento": "2026-05-10",
  "valor_total": 887.46,
  "transacoes": [
    {
      "data": "2026-04-22",
      "hora": "20:45:00",
      "descricao": "Supermercado Perola Da",
      "valor": -18.06,
      "categoria": "Mercado"
    },
    {
      "data": "2026-04-20",
      "hora": "17:01:00",
      "descricao": "Acaizeiros Resende",
      "valor": -44.00,
      "categoria": "Alimentação"
    },
    {
      "data": "2026-04-20",
      "hora": "13:59:00",
      "descricao": "53409490carolina",
      "valor": -35.47,
      "categoria": null
    },
    {
      "data": "2026-04-19",
      "hora": "16:59:00",
      "descricao": "Gelato Borelli",
      "valor": -49.00,
      "categoria": "Alimentação"
    },
    {
      "data": "2026-04-19",
      "hora": "14:34:00",
      "descricao": "Parada Bar",
      "valor": -67.50,
      "categoria": "Ali

In [7]:
import pymupdf # install with: pip install pymupdf
doc = pymupdf.open(source)
text = ""
for page in doc:
    text += page.get_text()


In [52]:
contagem = model.count_tokens(text)
print(f"O seu prompt consumirá: {contagem.total_tokens} tokens.")
response = model.generate_content(
    text,
    generation_config=configuracao
)
print(response.text)


O seu prompt consumirá: 3675 tokens.
{
  "documento_valido": true,
  "codigo_erro": null,
  "tipo_documento": "FATURA",
  "banco": "Sicredi",
  "data_vencimento": "2026-05-10",
  "valor_total": 887.46,
  "transacoes": [
    {
      "data": "2026-04-22",
      "hora": "20:45:00",
      "descricao": "Supermercado Perola Da",
      "valor": -18.06,
      "categoria": "Mercado"
    },
    {
      "data": "2026-04-20",
      "hora": "17:01:00",
      "descricao": "Acaizeiros Resende",
      "valor": -44.00,
      "categoria": "Restaurante"
    },
    {
      "data": "2026-04-20",
      "hora": "13:59:00",
      "descricao": "53409490carolina",
      "valor": -35.47,
      "categoria": null
    },
    {
      "data": "2026-04-19",
      "hora": "16:59:00",
      "descricao": "Gelato Borelli",
      "valor": -49.00,
      "categoria": "Restaurante"
    },
    {
      "data": "2026-04-19",
      "hora": "14:34:00",
      "descricao": "Parada Bar",
      "valor": -67.50,
      "categoria": "Res